<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w6_finance_rag/llm_260415_etf_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260415 ETF 데이터 수집, 전처리, 분석 + LLM 인사이트

**6주차 Day 2** | 금융 챗봇 프로젝트 - 데이터 파이프라인 구축

---

## 오늘 배울 내용

| 단계 | 내용 | 비유 |
|------|------|------|
| 1. 시계열 분해 | 트렌드 + 시즈널리티 + 잔차 | 기온 = 지구온난화(트렌드) + 여름/겨울(주기) + 나머지(잔차) |
| 2. 멀티 ETF 비교 | 여러 ETF 수익률/상관관계 분석 | 엑셀 시트 여러개 합치기 |
| 3. 투자 지표 계산 | 연간수익률, 변동성, MDD, 샤프 | 성적표 만들기 |
| 4. LLM 분석 연동 | 통계 지표를 LLM에 넘겨 인사이트 생성 | 데이터 분석가에게 보고서 부탁 |
| 5. Pandas 데이터 병합 | merge/concat/join | 엑셀 VLOOKUP |

---

### 전체 프로젝트 로드맵 (금융 챗봇)
```
1. 데이터 수집 + 전처리  <-- w6 Day1~2 (어제~오늘)
2. 데이터 분석 (LLM 연동) <-- w6 Day2 (오늘)
3. RAG용 데이터 생성      <-- w6 Day3~
4. RAG 구현 + 챗봇 완성   <-- 이후
```

## 0. 환경 설정

In [ ]:
!pip install -q finance-datareader langchain langchain-community langchain-core langchain-openai matplotlib numpy openai pandas scipy statsmodels

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import json
import random

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
# load_dotenv()

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'NanumGothic'  # Colab: 'NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

---
## 1. ETF 유니버스 탐색

> **비유**: ETF 유니버스 = 마트의 모든 상품 카테고리. 국내주식, 해외주식, 채권, 원자재 등 다양한 카테고리가 있다.

In [ ]:
# 한국 ETF 시장 카테고리별 종목 수 (대략)
categories = {
    "국내주식": 145, "해외주식": 120, "채권": 85,
    "섹터": 95, "원자재": 25, "부동산": 15, "기타": 30,
}

In [ ]:
# 카테고리별 분포 시각화 (파이차트 + 수평 바차트)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.pie(categories.values(), labels=categories.keys())
ax2.barh(list(categories.keys()), list(categories.values()))

plt.show()

In [ ]:
# 샘플 ETF 데이터 (수익률 vs 리스크)
etf_universe = [
    {"name": "KODEX 200", "cat": "국내주식", "ret_1y": 12.3, "risk": 15.2},
    {"name": "TIGER S&P500", "cat": "해외주식", "ret_1y": 18.5, "risk": 13.8},
    {"name": "KODEX 배당가치", "cat": "배당", "ret_1y": 8.2, "risk": 10.5},
    {"name": "TIGER 반도체", "cat": "섹터", "ret_1y": 35.2, "risk": 28.4},
    {"name": "KODEX 국고채3년", "cat": "채권", "ret_1y": 3.5, "risk": 2.1},
    {"name": "KODEX 골드선물", "cat": "원자재", "ret_1y": 15.8, "risk": 16.3},
    {"name": "KODEX 2차전지", "cat": "섹터", "ret_1y": -5.2, "risk": 32.1},
    {"name": "TIGER 단기통안채", "cat": "채권", "ret_1y": 3.2, "risk": 0.8},
]

df_etf = pd.DataFrame(etf_universe)
df_etf

### 리스크 vs 수익률 산점도

> **핵심**: 오른쪽 위 = 고수익 고위험, 왼쪽 아래 = 저수익 저위험. 이상적인 ETF는 **왼쪽 위** (저위험 고수익)에 있는 것.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# 카테고리별로 색 구분해서 scatter plot
for cat in df_etf['cat'].unique():
    sub = df_etf[df_etf['cat'] == cat]
    ax.scatter(sub['risk'], sub['ret_1y'], label=cat)
    for _, row in sub.iterrows():
        ax.annotate(row['name'][:8], (row['risk'], row['ret_1y']))

ax.set_xlabel('리스크 (%)')
ax.set_ylabel('1년 수익률 (%)')
ax.set_title('ETF 리스크-수익률 산점도')
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5)  # 기준선: 수익률 10%
ax.legend()
plt.show()

---
## 2. 데이터 스키마 설계 (dataclass)

> **비유**: dataclass = 이력서 양식. 어떤 항목을 채워야 하는지 미리 정해두는 것.
> `__post_init__`은 제출 전 검증 단계 -- 수수료가 음수면 반려!

**왜 필요한가?**
- 딕셔너리만 쓰면 오타(`expens_ratio`)를 잡을 수 없음
- dataclass는 타입 힌트 + 자동 검증으로 데이터 품질 보장
- `asdict()` / `json.dumps()`로 직렬화 -> LLM이나 DB에 넘기기 편함

In [ ]:
@dataclass
class ETFSchema:
    """ETF 기본 정보 스키마. dataclass는 __init__을 자동 생성해준다."""
    ticker: str            # 종목코드 (예: "069500")
    name: str              # 종목명
    category: str          # 카테고리
    expense_ratio: float   # 수수료 (%)
    risk_level: str = "중간"       # 기본값 = "중간"
    aum_billion: float = 0.0       # 순자산총액 (억원)
    description: str = ""          # 설명
    keywords: List[str] = field(default_factory=list)  # 키워드 목록

    def __post_init__(self):
        """생성 직후 자동 실행되는 검증 로직"""
        # 수수료 범위 검증 (0~5%)
        if self.expense_ratio < 0 or self.expense_ratio > 5:
            raise ValueError(f"수수료 범위 오류: {self.expense_ratio}")
        # 리스크 등급 검증
        valid_risks = ["매우낮음", "낮음", "중간", "높음", "매우높음"]
        if self.risk_level not in valid_risks:
            raise ValueError(f"리스크 등급 오류: {self.risk_level}")

In [ ]:
# 정상 생성 테스트
etf = ETFSchema(
    ticker="069500", name="KODEX 200", category="국내주식",
    expense_ratio=1.5, keywords=['코스피', '대형주', '인덱스']
)
etf

In [ ]:
# dataclass -> dict -> JSON (직렬화 체인)
# LLM에 데이터를 넘길 때, DB에 저장할 때 이 변환이 필수
etf_dict = asdict(etf)
etf_json = json.dumps(etf_dict, ensure_ascii=False, indent=2)
print(etf_json)

In [ ]:
# JSON -> dict -> dataclass (역직렬화)
# API에서 받은 데이터를 다시 객체로 복원
restored = ETFSchema(**json.loads(etf_json))
restored

In [ ]:
# 검증 테스트: 잘못된 데이터는 ValueError 발생
test_cases = [
    {"ticker": "A", "name": "정상", "category": "채권",
     "expense_ratio": 0.05, "risk_level": "낮음"},
    {"ticker": "B", "name": "수수료오류", "category": "주식",
     "expense_ratio": -0.5, "risk_level": "중간"},
    {"ticker": "C", "name": "리스크오류", "category": "원자재",
     "expense_ratio": 0.3, "risk_level": "최고"},  # "최고"는 유효하지 않음
]

for tc in test_cases:
    try:
        e = ETFSchema(**tc)
        print(f"  {tc['name']}: 생성 성공")
    except ValueError as err:
        print(f"  {tc['name']}: {err}")

### ETFReturns 스키마 (수익률 데이터용)

> `getattr(self, field_name)` = 문자열로 속성 접근. for문에서 여러 필드를 한꺼번에 검증할 때 유용.

In [ ]:
@dataclass
class ETFReturns:
    """ETF 수익률 스키마. 수익률 범위 자동 검증 (-100% ~ 500%)"""
    ticker: str
    return_1m: float
    return_3m: float
    return_1y: float
    return_3y: float

    def __post_init__(self):
        # 모든 수익률 필드를 루프로 검증 (getattr 활용)
        for field_name in ['return_1m', 'return_3m', 'return_1y', 'return_3y']:
            value = getattr(self, field_name)
            if value < -100 or value > 500:
                raise ValueError(f"{field_name} 오류: {value}")

# 테스트: 범위 밖의 값은 에러
# ETFReturns("123456", -200.1, 5.4, 12.3, 28.5)  # ValueError!

---
## 3. 실제 ETF 가격 데이터 수집 (FinanceDataReader)

> **비유**: `fdr.StockListing('ETF/KR')` = 마트에서 전체 상품 목록 가져오기, `fdr.DataReader(ticker)` = 특정 상품의 가격 이력 조회

FinanceDataReader는 한국 주식/ETF 데이터를 무료로 가져올 수 있는 라이브러리.

In [ ]:
# 한국 ETF 전체 목록
etf_list = fdr.StockListing('ETF/KR')
print(f"한국 ETF 총 {len(etf_list)}개")
etf_list.head(3)

In [ ]:
# KODEX 200 1년 가격 데이터 수집
ticker = "069500"
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

price_df = fdr.DataReader(ticker, start_date, end_date)
print(f"기간: {start_date} ~ {end_date}, 총 {len(price_df)}일")
price_df.head()

In [ ]:
# 가격 수집 함수 (에러 처리 포함)
def collect_etf_price(ticker, days=365):
    """ETF 가격 데이터를 수집. 실패 시 None 반환."""
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    try:
        df = fdr.DataReader(ticker, start_date, end_date)
        if len(df) > 0:
            return {'status': 'real', 'data': df, 'ticker': ticker}
    except Exception:
        pass

result = collect_etf_price('069500')
print(f"상태: {result['status']}, 데이터 수: {len(result['data'])}행")

---
## 4. 결측치/이상치 처리

### 결측치 처리 3가지 방법

> **비유**: 출석부에 빈칸이 있을 때
> - **Forward Fill**: 어제 출석했으니 오늘도 출석이겠지 (이전 값으로 채움)
> - **Interpolation**: 월요일 100원, 수요일 200원이면 화요일은 150원 (선형 보간)
> - **Rolling Mean**: 최근 5일 평균으로 채움 (이동평균)

주식 데이터는 주말/휴장일에 데이터가 없으므로 결측치 처리가 필수.

In [ ]:
# 인위적으로 5% 결측치 생성 (시뮬레이션용)
sample_df = price_df.copy()
mask = np.random.random(len(sample_df)) < 0.05
sample_df.loc[mask, 'Close'] = np.nan
print(f"결측치 개수: {sample_df['Close'].isna().sum()} / {len(sample_df)}")

In [ ]:
# 방법 1: Forward Fill (이전 값으로 채움)
ffill_df = sample_df.copy()
ffill_df['Close'] = ffill_df['Close'].ffill()

# 방법 2: Linear Interpolation (선형 보간)
interp_df = sample_df.copy()
interp_df['Close'] = interp_df['Close'].interpolate(method='linear')

# 방법 3: Rolling Mean (이동평균으로 채움)
rolling_df = sample_df.copy()
rolling_mean = rolling_df['Close'].rolling(5, min_periods=1).mean()
rolling_df['Close'] = rolling_df['Close'].fillna(rolling_mean)

print(f"ffill 결측: {ffill_df['Close'].isna().sum()}")
print(f"interp 결측: {interp_df['Close'].isna().sum()}")
print(f"rolling 결측: {rolling_df['Close'].isna().sum()}")

In [ ]:
# 3가지 방법 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, df_filled, title in zip(axes,
    [ffill_df, interp_df, rolling_df],
    ['Forward Fill', 'Interpolation', 'Rolling Mean']):
    ax.plot(sample_df.index, sample_df['Close'], 'ro', markersize=3, alpha=0.3, label='missing')
    ax.plot(df_filled.index, df_filled['Close'], label=title)
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 이상치 탐지 (Outlier Detection)

> **비유**: 시험 점수에서 0점이나 200점이 나오면 이상한 것처럼, 주가 데이터에서도 비정상적인 값을 찾아야 한다.

| 방법 | 기준 | 특징 |
|------|------|------|
| **IQR** | Q1 - 1.5*IQR ~ Q3 + 1.5*IQR | 분포에 강건, 비대칭 데이터에 적합 |
| **Z-score** | 평균 +/- 3*표준편차 | 정규분포 가정, 단순 |

In [ ]:
def detect_iqr(series, factor=1.5):
    """IQR 방식 이상치 탐지"""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - factor * iqr) | (series > q3 + factor * iqr)

def detect_z(series, threshold=3.0):
    """Z-score 방식 이상치 탐지"""
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

# 두 방법으로 이상치 개수 비교
z_outliers = detect_z(ffill_df)
iqr_outliers = detect_iqr(ffill_df)

print("Z-score 이상치:")
print(z_outliers.sum())
print("\nIQR 이상치:")
print(iqr_outliers.sum())
print("\n교집합 (둘 다 이상치):")
print((z_outliers & iqr_outliers).sum())

---
## 5. 피처 엔지니어링 (수익률, 변동성, 로그 수익률)

> **비유**:
> - `pct_change()` = 어제 대비 오늘 몇 % 올랐는지 (일간 수익률)
> - `rolling(5).std()` = 최근 5일간 수익률이 얼마나 출렁였는지 (변동성)
> - `log_return` = 복리 수익률 계산용 (수학적으로 더 정확)

In [ ]:
features = ffill_df[['Close']].copy()

# 일간 수익률
features['return'] = features['Close'].pct_change()

# 변동성 (5일, 20일 윈도우)
features['vol_5d'] = features['return'].rolling(5).std()
features['vol_20d'] = features['return'].rolling(20).std()

# 로그 수익률 (복리 계산에 적합)
features['log_return'] = np.log(features['Close'] / features['Close'].shift(1))

features.head(10)

### 전처리 파이프라인 함수

> 결측치 처리 -> 이상치 처리 -> 피처 생성을 하나의 함수로 묶어두면 재사용이 편리하다.

In [ ]:
def preprocess_pipeline(df):
    """가격 데이터 전처리 파이프라인"""
    result = df[['Close']].copy()

    # 1. 결측값 처리 (interpolation -> ffill -> bfill)
    result['Close'] = result['Close'].interpolate(method='linear')
    result['Close'] = result['Close'].ffill().bfill()

    # 2. 이상치를 ffill (IQR * 1.5 기준)
    ret = result['Close']
    q1, q3 = ret.quantile(0.25), ret.quantile(0.75)
    iqr = q3 - q1
    outlier_mask = (ret < q1 - 1.5 * iqr) | (ret > q3 + 1.5 * iqr)
    result.loc[outlier_mask, 'Close'] = np.nan
    result['Close'] = result['Close'].ffill()

    # 3. 피처 추가
    result['return'] = result['Close'].pct_change()
    result['log_return'] = np.log(result['Close'] / result['Close'].shift(1))

    return result

processed = preprocess_pipeline(price_df)
processed.head()

---
## 6. 이동평균 + 트렌드 분해 (Time Series Decomposition)

> **핵심 개념**: 시계열 데이터 = 트렌드 + 시즈널리티 + 잔차
>
> **비유** (강사 설명): 기온 데이터를 예로 들면
> - **트렌드**: 지구온난화로 전반적 온도가 조금씩 높아지는 장기적 추세
> - **시즈널리티**: 여름에 높고 겨울에 낮은 반복 주기
> - **잔차(Residual)**: 트렌드와 주기로 설명할 수 없는 나머지
>
> **Additive 모델**: 원본 = 트렌드 + 시즈널리티 + 잔차 (값을 더함)
> **Multiplicative 모델**: 원본 = 트렌드 x 시즈널리티 + 잔차 (값을 곱함)
>
> 좋은 모델일수록 잔차가 작다 = 트렌드와 주기만으로 잘 설명된다.

### SMA vs EMA
- **SMA (Simple Moving Average)**: 최근 N일 단순 평균
- **EMA (Exponential MA)**: 최근 값에 더 큰 가중치 -> 변화에 빠르게 반응

In [ ]:
close = ffill_df['Close']

# 이동평균 계산
sma_20 = close.rolling(20).mean()    # 단순 이동평균 (20일)
ema_20 = close.ewm(span=20).mean()   # 지수 이동평균 (20일) - ewm = Exponential Weighted Method

# 트렌드 분해: 주간 데이터로 리샘플링 후 분해
weekly = close.resample('W').last().dropna()   # 주단위 마지막 값
decomp = seasonal_decompose(weekly, model='additive', period=13)  # 13주 = 분기

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# 상단: 이동평균 비교
ax = axes[0]
ax.plot(close.index, close, alpha=0.4, linewidth=0.5, label='원본(original)')
ax.plot(sma_20.index, sma_20, linewidth=1.5, label='SMA(20)', color='red')
ax.plot(ema_20.index, ema_20, linewidth=1.5, label='EMA(20)', color='green')
ax.set_title('이동평균 비교 (SMA vs EMA)')
ax.legend()

# 하단: 트렌드 분해 결과
axes[1].plot(decomp.trend.index, decomp.trend, label='Trend', linewidth=1.5, color='blue')
axes[1].plot(decomp.seasonal.index, decomp.seasonal, label='Seasonality', linewidth=1.5, color='red')
axes[1].set_title('트렌드 분해 (Trend + Seasonality)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. 멀티 ETF 비교 분석 (수익률, 누적수익률, 상관관계)

> **비유**: 여러 학생의 시험 성적을 한 표에 모아서 비교하는 것.
> - 누적수익률 = 시험 점수 누적 합산
> - 상관관계 = A학생이 잘 볼 때 B학생도 잘 보는지?

In [ ]:
# 비교할 ETF 4종
tickers = {
    '069500': 'KODEX 200 (국내주식)',
    '360750': 'TIGER S&P500 (미국주식)',
    '152380': 'KODEX 국고채10년 (채권)',
    '132030': 'KODEX 골드선물 (원자재)',
}

# 각 ETF의 종가를 하나의 DataFrame으로 합치기
multi_close = pd.DataFrame()
for ticker, name in tickers.items():
    df = fdr.DataReader(ticker, start=(datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d'))
    multi_close[name] = df['Close']

multi_close = multi_close.dropna()  # 결측치 제거
multi_close.head()

In [ ]:
# 일간 수익률 + 누적 수익률 계산
returns = multi_close.pct_change().dropna()

# 누적수익률 = (1 + 일간수익률)을 계속 곱한 것
# 비유: 100만원이 매일 1% 오르면 -> 1.01 * 1.01 * 1.01 * ... = 복리 효과
cum_returns = (1 + returns).cumprod()

print("=== 최종 누적수익률 ===")
cum_returns.tail(1)

In [ ]:
# 누적수익률 비교 그래프
fig, ax = plt.subplots(figsize=(12, 5))
for col in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[col], label=col, linewidth=1.2)

ax.axhline(1.0, color='black')  # 기준선 (원금)
ax.legend(fontsize=8)
ax.set_title('누적 수익률 비교 (Cumulative Returns)')
plt.show()

### 상관관계 분석 (Pearson Correlation)

> **비유** (강사 설명): "KODEX 200이 100원 오를 때 TIGER S&P500이 얼마 오르는지"
> - 1.0 = 완벽한 양의 상관 (같이 오르내림)
> - 0.0 = 상관 없음
> - -1.0 = 완벽한 음의 상관 (하나 오르면 다른 건 내림)
>
> **분산투자 핵심**: 상관관계가 낮은 자산끼리 섞으면 리스크가 줄어든다!

In [ ]:
# 상관관계 매트릭스
corr_matrix = returns.corr()
corr_matrix

In [ ]:
# 히트맵 + 롤링 상관관계 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 좌측: 상관관계 히트맵
im = ax1.imshow(corr_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax1.set_xticks(range(len(corr_matrix)))
ax1.set_yticks(range(len(corr_matrix)))
short_names = [c.split('(')[0].strip() for c in corr_matrix.columns]
ax1.set_xticklabels(short_names, rotation=45)
ax1.set_yticklabels(short_names, fontsize=8)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax1.text(j, i, f'{corr_matrix.iloc[i,j]:.2f}',
                 fontsize=10, ha='center', va='center')
fig.colorbar(im, ax=ax1)
ax1.set_title('Correlation Matrix (히트맵)')

# 우측: 60일 롤링 상관관계
# 국내주식 vs 미국주식의 상관관계가 시간에 따라 어떻게 변하는지
cols = returns.columns.tolist()
if len(cols) >= 2:
    rolling_corr = returns[cols[0]].rolling(60).corr(returns[cols[1]])
    ax2.plot(rolling_corr.index, rolling_corr, linewidth=0.8)
    ax2.set_title(f'60일 롤링 상관관계: {short_names[0]} vs {short_names[1]}')

plt.tight_layout()
plt.show()

---
## 8. 투자 지표 계산 (연간수익률, 변동성, MDD, 샤프비율)

> 주식 거래일 = 1년에 약 **252일** (365일 - 주말 - 공휴일)
>
> | 지표 | 의미 | 공식 | 좋은 값 |
> |------|------|------|--------|
> | **연간수익률** | 1년간 얼마 벌었나 | `(1+일간평균)^252 - 1` | 높을수록 좋음 |
> | **연간변동성** | 수익률이 얼마나 출렁이나 | `일간표준편차 * sqrt(252)` | 낮을수록 안정적 |
> | **MDD** | 최고점 대비 최대 낙폭 | `최저점/최고점 - 1` | 0에 가까울수록 좋음 |
> | **샤프비율** | 위험 대비 수익 | `(수익률-무위험)/변동성` | 1 이상이면 우수 |

In [ ]:
def calculate_metrics(returns_series, risk_free=0.035):
    """투자 핵심 지표 4가지 계산"""
    # 연간 수익률: 일간 평균 수익률을 252일 복리로
    ann_ret = (1 + returns_series.mean()) ** 252 - 1
    # 연간 변동성: 일간 표준편차에 sqrt(252) 곱하기
    ann_vol = returns_series.std() * np.sqrt(252)
    # 샤프 비율: (수익률 - 무위험수익률) / 변동성
    sharpe = (ann_ret - risk_free) / ann_vol
    # MDD: 누적 수익 곡선에서 최고점 대비 최대 하락폭
    cum = (1 + returns_series).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()

    return {
        'yearly return': f'{ann_ret * 100:.1f}%',
        'yearly volatile': f'{ann_vol * 100:.1f}%',
        'MDD': f'{mdd * 100:.1f}%',
        'sharpe': f'{sharpe:.1f}'
    }

In [ ]:
# 모든 ETF에 대해 지표 계산
metrics_rows = []
for col in returns.columns:
    m = calculate_metrics(returns[col])
    m['ETF'] = col.split('(')[0].strip()
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index('ETF')
metrics_df

---
## 9. LLM 연동: 데이터 분석 인사이트 생성

> **핵심 패턴**: 통계 지표를 계산한 후 -> LLM에 넘겨서 -> 자연어 분석 리포트를 생성
>
> **왜 직접 계산 후 LLM에 넘기나?**
> - LLM에게 raw 데이터(수백 개 값)를 통째로 넘기면 토큰 낭비 + 부정확
> - 통계량(평균, 분산 등)을 먼저 계산해서 요약값만 넘기면 정확 + 저렴
> - 비유: 회계사에게 영수증 1000장을 주는 것 vs 결산표를 주는 것

### 9-1. 방법 A: 지표 테이블을 LLM에 넘기기

In [ ]:
def generate_analysis_insight(metrics_df):
    """투자 지표 테이블을 LLM에 넘겨 인사이트 생성"""
    response = ChatOpenAI(model='gpt-4o-mini', temperature=0.3, max_tokens=400).invoke([
        {'role': 'system', 'content': '당신은 금융 데이터 분석가입니다. 수치 기반으로 직관적인 인사이트를 제공합니다'},
        {'role': 'user', 'content': f"""아래 ETF 비교 데이터를 분석하여 투자 인사이트를 작성하세요.

{metrics_df.to_string()}

다음 형식으로 답하세요:
1. 핵심 발견(3줄)
2. 위험 요인(2줄)
3. 분산 투자 제안(2줄)"""}
    ])
    return response.content

insight = generate_analysis_insight(metrics_df)
print(insight)

### 9-2. 방법 B: 상세 통계량을 직접 계산 후 LLM에 넘기기

> 이 방법이 더 정확한 분석을 만들어낸다. LLM이 계산을 안 해도 되니까.
>
> **Skewness (왜도)**: 분포가 좌우로 얼마나 치우쳤는지
> - 0 = 정규분포, 양수 = 왼쪽 치우침, 음수 = 오른쪽 치우침
>
> **Kurtosis (첨도)**: 분포가 얼마나 뾰족한지
> - 정규분포 = 3, 3보다 크면 뾰족 (꼬리가 두꺼움), 3보다 작으면 납작

### 9-3. 방법 C: LLM에 raw 데이터만 넘기기 (비교용)

> 통계 계산 없이 최근 가격 120개를 통째로 넘김. 토큰을 많이 쓰고 분석 품질이 떨어진다.

In [ ]:
def generate_analysis_report_llm_only(ticker_name, close_prices):
    """LLM에 raw 가격 데이터를 직접 넘기는 방식 (비교용)"""
    recent_prices = close_prices.tail(120).tolist()

    prompt = f"""
    다음은 {ticker_name}의 종가 데이터입니다.

    {recent_prices}

    이 데이터를 기반으로
    - 추세 (상승/하락/횡보)
    - 변동성 특징
    - 최근 이상 움직임 여부

    를 판단하고 2~3문장으로 요약하세요.

    수치 계산은 직접 수행해서 근거 기반으로 설명하세요.
    투자 추천은 하지 마세요. 객관적 현황만 서술해주세요."""

    response = ChatOpenAI(model='gpt-4o-mini', temperature=0.5, max_tokens=400).invoke([
        {'role': 'system', 'content': '당신은 데이터 분석가입니다. 수치에 근거해서 분석하세요'},
        {'role': 'user', 'content': prompt}
    ])

    return response.content

report_llm = generate_analysis_report_llm_only('KODEX 200', close)
print(report_llm)

---
## 10. Pandas 데이터 병합 (merge, concat, join)

> **비유**: 엑셀의 VLOOKUP과 같은 개념
>
> | 함수 | 역할 | SQL 대응 | 비유 |
> |------|------|----------|------|
> | `pd.concat()` | 행/열 방향으로 붙이기 | UNION | 엑셀 시트 이어붙이기 |
> | `pd.merge()` | 키 기준으로 합치기 | JOIN | 엑셀 VLOOKUP |
>
> ### Join 종류
> - **inner**: 교집합 (양쪽에 다 있는 것만)
> - **outer**: 합집합 (없는 건 NaN)
> - **left**: 왼쪽 테이블 기준
> - **right**: 오른쪽 테이블 기준

In [ ]:
# 최근 6개월 데이터 수집
kodex = fdr.DataReader('069500', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
tiger = fdr.DataReader('360750', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
bond = fdr.DataReader('152380', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))

In [ ]:
# concat으로 여러 ETF 종가 합치기 (rename으로 컬럼명 지정)
combined = pd.concat([
    kodex['Close'].rename('kodex_close'),
    tiger['Close'].rename('tiger_close'),
    bond['Close'].rename('bond_close')
], axis=1)  # axis=1 = 옆으로(열방향) 붙이기

combined.head()

In [ ]:
# merge 예시: 메타데이터 + 분석결과를 ticker 기준으로 합치기
meta = pd.DataFrame([
    {'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense': 0.15},
    {'ticker': '360750', 'name': 'TIGER S&P500', 'category': '해외주식', 'expense': 0.07},
    {'ticker': '152380', 'name': 'KODEX 국고채10년', 'category': '채권', 'expense': 0.05},
    {'ticker': '132030', 'name': 'KODEX 골드선물', 'category': '원자재', 'expense': 0.68},
])

analysis = pd.DataFrame([
    {'ticker': '069500', 'sharpe': 0.82, 'mdd': -0.15},
    {'ticker': '360750', 'sharpe': 1.25, 'mdd': -0.12},
    {'ticker': '152380', 'sharpe': 0.31, 'mdd': -0.05},
    {'ticker': '132030', 'sharpe': 0.95, 'mdd': -0.08},
])

# inner join: ticker가 양쪽에 모두 있는 행만 합침
merged = pd.merge(meta, analysis, on='ticker', how='inner')
merged

### 결측 날짜 처리: reindex + ffill

> ETF마다 거래일이 다를 수 있다. B에 없는 날짜는 A의 인덱스로 reindex한 후 ffill로 채운다.

In [ ]:
# A와 B 시리즈 준비 (B에서 랜덤하게 5개 날짜 제거)
df_a = kodex['Close'].rename('A')
df_b = tiger['Close'].rename('B')

np.random.seed(42)
drop_idx = np.random.choice(df_b.index, size=5, replace=False)  # 비복원 추출
df_b_missing = df_b.drop(drop_idx)

print(f"A: {len(df_a)}행, B(missing): {len(df_b_missing)}행")

In [ ]:
# concat + dropna = inner join 효과
inner = pd.concat([df_a, df_b_missing], axis=1).dropna()
print(f"Inner join 결과: {len(inner)}행")

# reindex + ffill = outer join + 결측 채우기
aligned = df_b_missing.reindex(df_a.index).ffill()
outer = pd.concat([df_a, aligned.rename('B_aligned')], axis=1)
print(f"Outer (reindex+ffill) 결과: {len(outer)}행")
outer.head()

### 종합 병합 함수: merge_all

> 가격 데이터 딕셔너리 + 메타데이터를 받아서 연간수익률을 계산하고 merge한 최종 테이블을 반환.

In [ ]:
def merge_all(price_dict, meta_df):
    """가격 데이터 + 메타데이터를 merge하여 최종 분석 테이블 반환"""
    # 1. 모든 ticker의 Close를 하나의 DataFrame으로
    closes = pd.concat(
        {ticker: df['Close'] for ticker, df in price_dict.items()},
        axis=1
    ).dropna()

    # 2. ticker별 연간 수익률 계산
    analysis_rows = []
    for ticker in closes.columns:
        ret = closes[ticker].pct_change().dropna()
        ann_ret = (1 + ret.mean()) ** 252 - 1  # 252 거래일 복리
        analysis_rows.append({
            'ticker': ticker,
            'annual_return': round(ann_ret * 100, 2)
        })
    analysis_df = pd.DataFrame(analysis_rows)

    # 3. 메타데이터와 merge (inner join)
    result = pd.merge(meta_df, analysis_df, on='ticker', how='inner')
    return result

In [ ]:
# 실행: 3개 ETF의 최종 분석 테이블
price_dict = {
    '069500': kodex,
    '360750': tiger,
    '152380': bond,
}

result = merge_all(price_dict, meta)
result

---
## 핵심 정리

### 오늘 배운 것

1. **dataclass**: 데이터 스키마 정의 + 자동 검증 (`__post_init__`) + 직렬화 (`asdict` -> `json.dumps`)
2. **결측치 처리**: ffill, interpolation, rolling mean 3가지 방법
3. **이상치 탐지**: IQR 방식 vs Z-score 방식
4. **시계열 분해**: 트렌드(장기추세) + 시즈널리티(주기) + 잔차
5. **투자 지표**: 연간수익률, 변동성, MDD, 샤프비율 (252 거래일 기준)
6. **LLM 연동**: 직접 계산한 통계량을 LLM에 넘기면 더 정확한 분석
7. **Pandas 병합**: concat(이어붙이기) vs merge(키기준 합치기) vs reindex(인덱스 맞추기)

### 프로젝트 진행 상황
```
[완료] 데이터 수집 (FinanceDataReader)
[완료] 전처리 (결측치/이상치/피처 엔지니어링)
[완료] 분석 (시계열 분해, 상관관계, 투자지표, LLM 인사이트)
[다음] RAG용 데이터 생성 -> RAG 구현 -> 금융 챗봇 완성
```